# 기본 EDA 실습 — Palmer Penguins

이번에는 모델링에 앞서 **EDA(탐색적 데이터 분석)**만 집중적으로 연습해보겠습니다.

사용할 데이터는 남극 팔머 군도에 서식하는 펭귄 관측 데이터인 **Palmer Penguins**입니다. seaborn에 내장되어 있어서 별도 다운로드 없이 바로 불러올 수 있습니다.

### 데이터 기본 정보

| 컬럼 | 뜻 |
|---|---|
| `species` | 펭귄 종 (Adelie / Chinstrap / Gentoo) |
| `island` | 서식 섬 (Torgersen / Biscoe / Dream) |
| `bill_length_mm` | 부리 길이 (mm) |
| `bill_depth_mm` | 부리 두께 (mm) |
| `flipper_length_mm` | 지느러미 길이 (mm) |
| `body_mass_g` | 몸무게 (g) |
| `sex` | 성별 (Male / Female) |
| `year` | 관측 연도 |

EDA의 목표는 "모델을 만드는 것"이 아니라 **데이터가 어떻게 생겼는지, 어떤 패턴과 문제가 있는지를 파악하는 것**입니다. 순서대로 하나씩 살펴보겠습니다.

## 1. 데이터 불러오기 & 첫인상

먼저 데이터를 불러와서 어떻게 생겼는지 눈으로 확인해보겠습니다.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset("penguins")
df.head()

In [ ]:
print(df.shape)
df.info()

344개 행, 7개 컬럼이 있네요.

`info()`의 Dtype을 보면 `species`, `island`, `sex`는 object(문자열), 나머지는 수치형(float64/int64)입니다.

Non-Null Count를 보면 `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`가 344보다 적은 걸 볼 수 있습니다. 즉 결측치가 있다는 뜻이죠. 뒤에서 자세히 확인해보겠습니다.

## 2. 기초 통계 확인

수치형 변수는 `describe()`로, 범주형 변수는 `value_counts()`로 분포를 확인합니다.

In [ ]:
df.describe()

수치형 컬럼들의 평균, 표준편차, 최솟값/최댓값 등을 한눈에 볼 수 있습니다.

예를 들어 `body_mass_g`는 최소 2700g에서 최대 6300g까지 범위가 꽤 넓네요. 종에 따라 몸집 차이가 클 수 있다는 걸 짐작할 수 있습니다.

이번엔 범주형 컬럼들을 하나씩 살펴보겠습니다.

In [ ]:
for col in ['species', 'island', 'sex']:
    print(f"[{col}]")
    print(df[col].value_counts(dropna=False))
    print()

`species`는 Adelie가 가장 많고 Chinstrap이 가장 적어서 약간 불균형합니다.

`sex`에는 `NaN`이 몇 개 섞여 있는 게 보이네요. 결측치를 좀 더 자세히 확인해보겠습니다.

## 3. 결측치 확인

컬럼별로 결측치가 몇 개나 있는지, 그리고 어떤 행에 몰려있는지 확인해보겠습니다.

In [ ]:
df.isna().sum()

In [ ]:
# 결측치가 하나라도 있는 행만 확인
df[df.isna().any(axis=1)]

결측치가 있는 행들을 보면, `bill_length_mm`부터 `body_mass_g`까지 한꺼번에 비어있는 행이 대부분입니다. 아마 해당 개체를 측정 자체를 못 한 경우로 보입니다.

반면 `sex`만 따로 비어있는 행들도 있는데, 이건 성별 판별이 어려웠던 경우일 수 있습니다.

이렇게 **"왜 비어있는가"**를 생각해보는 게 결측치 처리 방향을 정하는 데 중요합니다. 지금 실습에서는 시각화 위주로 진행할 것이기 때문에 결측치를 그대로 두고, 필요한 시점에만 제외하고 보도록 하겠습니다.

## 4. 단변량 분포 시각화

변수 하나씩 분포를 살펴보겠습니다. 먼저 수치형 변수들의 히스토그램입니다.

In [ ]:
num_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, col in zip(axes.flatten(), num_cols):
    ax.hist(df[col].dropna(), bins=20)
    ax.set_title(col)
plt.tight_layout()
plt.show()

`bill_length_mm`이나 `flipper_length_mm`은 봉우리가 두 개인 것처럼 보이기도 합니다. 이건 종(species)이 섞여 있어서 그런 걸로 추측해볼 수 있습니다. 뒤에서 종별로 나눠서 다시 확인해보겠습니다.

이번엔 범주형 변수들의 분포를 막대그래프로 확인합니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col in zip(axes, ['species', 'island', 'sex']):
    df[col].value_counts(dropna=False).plot(kind='bar', ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 5. 그룹별 비교 — 종(species)에 따라 신체 특성이 다를까?

앞서 히스토그램에서 본 두 개의 봉우리가 정말 종 차이 때문인지, `species`별로 나눠서 박스플롯으로 확인해보겠습니다.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, col in zip(axes.flatten(), num_cols):
    sns.boxplot(data=df, x='species', y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

역시 종별로 뚜렷한 차이가 보입니다. 특히 `flipper_length_mm`과 `body_mass_g`는 Gentoo가 확실히 크고, `bill_depth_mm`은 오히려 Gentoo가 더 작네요.

숫자로도 확인해보겠습니다.

In [ ]:
df.groupby('species')[num_cols].mean()

표로 보니 차이가 더 명확하네요. 이 정도면 신체 사이즈만으로도 종을 어느 정도 구분할 수 있을 것 같습니다.

## 6. 범주형 변수 간 관계

`species`와 `island`, `species`와 `sex`가 서로 관련이 있는지 교차표로 살펴보겠습니다.

In [ ]:
pd.crosstab(df['species'], df['island'])

흥미로운 패턴이 보입니다. Adelie는 세 섬 모두에서 관찰되지만, Chinstrap은 Dream 섬에서만, Gentoo는 Biscoe 섬에서만 관찰됩니다. 즉 서식 섬만 알아도 종을 어느 정도 좁힐 수 있다는 뜻이죠.

In [ ]:
pd.crosstab(df['species'], df['sex'])

`sex`는 종과 큰 관련 없이 대체로 균등하게 분포되어 있네요.

## 7. 수치형 변수 간 관계

수치형 변수들끼리는 서로 얼마나 관련이 있는지 상관관계 히트맵으로 확인해보겠습니다.

In [ ]:
corr = df[num_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("수치형 변수 간 상관관계")
plt.show()

`flipper_length_mm`과 `body_mass_g`는 상관관계가 매우 높습니다 (지느러미가 길수록 몸무게도 많이 나감). 반면 `bill_depth_mm`은 나머지 변수들과 음의 상관관계를 보이는 게 특이하네요.

이번엔 `species`로 색을 구분해서 산점도 행렬(pairplot)로 한 번에 살펴보겠습니다.

In [ ]:
sns.pairplot(df, hue='species', vars=num_cols)
plt.show()

전체 상관관계 히트맵에서는 잘 안 보였던 패턴이, `species`로 나눠서 보니 확실히 드러납니다. 특히 `bill_length_mm`과 `bill_depth_mm`을 같이 보면 세 종이 거의 겹치지 않고 군집을 이루는 걸 볼 수 있습니다.

즉 앞서 히트맵에서 `bill_depth_mm`이 이상하게 음의 상관을 보였던 이유도, 사실은 종 내에서는 양의 상관인데 종을 섞어서 보니 반대로 나타난 것(심슨의 역설과 비슷한 상황)일 수 있습니다.

## 8. 정리

지금까지 살펴본 내용을 정리해보겠습니다.

- 데이터에는 344개 관측치, 7개 컬럼이 있고 일부 수치형 컬럼과 `sex`에 결측치가 존재합니다.
- `species`별로 신체 사이즈(특히 `flipper_length_mm`, `body_mass_g`, `bill_depth_mm`)가 뚜렷하게 다릅니다.
- `island`도 종과 강하게 연관되어 있어서, 섬 정보만으로도 종을 어느 정도 좁힐 수 있습니다.
- 변수 간 상관관계는 전체로 볼 때와 종별로 나눠서 볼 때 다르게 나타날 수 있다는 점도 확인했습니다 (그룹을 무시하고 상관관계를 해석하면 잘못된 결론에 이를 수 있음).

**생각해볼 질문:** 만약 이 데이터로 종을 분류하는 모델을 만든다면, 가장 유용해 보이는 특성은 무엇일까요? 그리고 어떤 특성은 크게 도움이 안 될 것 같나요?